In [0]:
%sql

-- Question 4: Do traffic advisories correlate with changes in pickup volume or trip duration within impacted boroughs?
-- Suggested chart: Grouped bar | X: borough | Y: avg_duration_minutes | Color/Group: advisory_type

WITH trip_advisory AS (
    SELECT 
        t.pu_location_id,
        t.do_location_id,
        t.pickup_datetime,
        t.dropoff_datetime,
        z.borough,
        t.trip_duration_min,
        t.trip_distance,
        t.total_amount,
        a.advisory_type,
        ROW_NUMBER() OVER (
            PARTITION BY t.pu_location_id, t.do_location_id, t.pickup_datetime, t.dropoff_datetime
            ORDER BY a.effective_from DESC
        ) AS advisory_rank
    FROM nyc_mobility.mart.fact_trip AS t
    JOIN nyc_mobility.mart.dim_zone AS z
        ON t.pu_location_id = z.location_id
    LEFT JOIN nyc_mobility.mart.dim_advisory AS a
        ON LOWER(TRIM(a.borough)) = LOWER(TRIM(z.borough))
       AND t.pickup_datetime BETWEEN a.effective_from AND a.effective_to
)
SELECT 
    borough,
    COALESCE(advisory_type, 'None') AS advisory_type,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_duration_min), 2) AS avg_duration_minutes,
    ROUND(AVG(total_amount), 2) AS avg_total_fare
FROM trip_advisory
WHERE advisory_rank = 1
GROUP BY borough, advisory_type
ORDER BY borough, advisory_type;